# Topic 40 — Model Interpretation
### ⭐ Important for research. Theory → feature importance → permutation importance → SHAP → LIME (text).

A model that performs well but can't be explained is hard to trust, debug, or write about
convincingly in a paper. **Interpretability/explainability** tools tell you WHY a model made a
particular prediction, or which features matter most overall — essential for a cyberbullying
detection paper where reviewers will ask "what is the model actually picking up on?".

In [ ]:
!pip install shap lime -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
import shap
from lime.lime_text import LimeTextExplainer

rng = np.random.default_rng(0)

## 1. Built-in feature importance (tree models — reused from Topic 13)

Random Forest's `feature_importances_` measures how much each feature reduced impurity (Topic 12)
across all trees, on average. Fast, but can be misleading for correlated/high-cardinality features.

In [ ]:
X, y = make_classification(n_samples=300, n_features=6, n_informative=3, random_state=42)
feature_names = [f"feature_{i}" for i in range(6)]

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)

plt.figure(figsize=(6, 4))
plt.bar(feature_names, rf.feature_importances_)
plt.xticks(rotation=45)
plt.title("Built-in Random Forest feature importance")
plt.show()

## 2. Permutation importance — a more reliable, model-agnostic alternative

Idea: shuffle (permute) ONE feature's values across all samples, breaking its relationship with
the target, and see how much the model's performance DROPS. A big drop = that feature mattered a
lot. Works for ANY model (not just trees), and directly measures impact on actual predictive
performance, not just an internal training statistic.

In [ ]:
result = permutation_importance(rf, X, y, n_repeats=10, random_state=42, scoring="f1")

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std,
}).sort_values("importance_mean", ascending=False)

plt.figure(figsize=(6, 4))
plt.barh(importance_df["feature"], importance_df["importance_mean"], xerr=importance_df["importance_std"])
plt.title("Permutation importance (with std error bars across repeats)")
plt.gca().invert_yaxis()
plt.show()
print(importance_df)

## 3. SHAP — theoretically grounded, per-prediction explanations

**SHAP** (SHapley Additive exPlanations) is based on game theory: it fairly distributes "credit"
for a prediction among all input features, guaranteeing some nice mathematical properties
(consistency, local accuracy). Gives you BOTH:
- **Global** importance (average impact across all predictions), and
- **Local** explanations (why THIS specific prediction came out this way).

In [ ]:
explainer = shap.TreeExplainer(rf)      # TreeExplainer is fast, exact, for tree-based models
shap_values = explainer.shap_values(X)

# shap_values shape depends on sklearn/shap version -- handle both binary-classification formats
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]     # SHAP values for predicting class 1
else:
    shap_values_class1 = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

shap.summary_plot(shap_values_class1, X, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()
# Each dot = one sample. Position on x-axis = how much that feature pushed THIS prediction
# toward class 1 (right) or class 0 (left). Color = the feature's actual value (red=high, blue=low).

In [ ]:
# Local explanation: why did the model predict THIS specific sample the way it did?
sample_idx = 0
print("sample features:", X[sample_idx])
print("model's prediction:", rf.predict(X[sample_idx:sample_idx+1])[0])
print("\nSHAP contribution of each feature to THIS prediction:")
for name, val in zip(feature_names, shap_values_class1[sample_idx]):
    print(f"  {name}: {val:+.3f}")
# Positive values pushed toward class 1, negative pushed toward class 0 -- these sum up
# (plus a baseline) to explain exactly how the model arrived at this one prediction.

## 4. LIME — model-agnostic, works on ANY model (including text!)

**LIME** (Local Interpretable Model-agnostic Explanations) explains one prediction at a time by
training a small, simple, interpretable model (usually linear) that approximates the complex
model's behavior JUST in the local neighborhood around that one prediction. Works even on models
SHAP's fast tree-explainer can't handle, including full text classification pipelines.

In [ ]:
bullying_examples = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "you should just disappear", "you are pathetic",
]
not_bullying_examples = [
    "great job today team", "have a wonderful day", "nice work everyone",
    "thanks for your help", "well done on the project", "excellent effort today",
]
texts = bullying_examples + not_bullying_examples
y_text = np.array([1]*6 + [0]*6)

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
text_pipeline.fit(texts, y_text)

explainer_lime = LimeTextExplainer(class_names=["not_bullying", "bullying"])

sample_text = "you are worthless and stupid"
explanation = explainer_lime.explain_instance(
    sample_text, text_pipeline.predict_proba, num_features=6
)

print(f"explaining prediction for: '{sample_text}'")
print("\nword contributions:")
for word, weight in explanation.as_list():
    direction = "-> bullying" if weight > 0 else "-> not_bullying"
    print(f"  {word:<12} weight={weight:+.3f}  {direction}")

In [ ]:
# LIME can also render an inline visualization
explanation.show_in_notebook(text=sample_text)

## 5. Which tool when?

| Tool | Best for | Speed |
|---|---|---|
| Built-in `feature_importances_` | Quick first look, tree models only | Fastest |
| Permutation importance | Model-agnostic, more trustworthy than built-in | Medium |
| SHAP | Rigorous, both global + local, great plots | Slower (fast for trees, slower for other models) |
| LIME | ANY model including full text pipelines, per-prediction | Medium (one prediction at a time) |

For your cyberbullying paper: SHAP or LIME's per-word contributions (like the LIME output above)
are exactly the kind of evidence a paper would show to argue "the model is picking up on genuinely
toxic language, not spurious correlations."

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Run explainer_lime.explain_instance on 3 more sentences of your own and inspect which words
#    get flagged as most influential.
# 2. Compare permutation_importance's ranking to rf.feature_importances_'s ranking on the same
#    RandomForestClassifier -- are the top features the same?
# 3. Try shap.KernelExplainer (works for ANY model, slower than TreeExplainer) on the LogisticRegression
#    text_pipeline instead of LIME, and compare its word-level explanation to LIME's.
# 4. Once you have real predictions on your cyberbullying dataset, pick 3 FALSE POSITIVES
#    (model wrongly said "bullying") and use LIME to see which words misled it -- this is
#    valuable error analysis material for your paper's discussion section.

---
### Next up: **Topic 41 — Experimentation & Reproducibility**.

Say "next" when you're ready.